# 11. Stream Wrappers for Dynamic Feature Spaces

In standard stream learning, every instance arrives with the **same fixed set of features**.
Real-world deployments often break this assumption: sensors go offline, new signals are added
mid-deployment, or feature sets rotate on a schedule.

OpenMOA provides five **Stream Wrappers** that wrap any static dataset and simulate
feature-space evolution over time:

| Wrapper | Paradigm | Output dimensionality |
|---------|----------|-----------------------|
| `OpenFeatureStream` | Variable-length with 6 evolution patterns | Variable (attaches `feature_indices`) |
| `TrapezoidalStream` | Features phase in gradually (TDS) | Fixed (inactive → NaN) |
| `CapriciousStream` | Features missing at random (CDS) | Fixed (missing → NaN) |
| `EvolvableStream` | Features rotate in segments (EDS) | Fixed (inactive → NaN) |
| `ShuffledStream` | Eliminates label-sorting bias | Fixed (unchanged) |

This notebook walks through each wrapper with working examples and visualisations.

In [ ]:
# This cell is hidden on openmoa.net. See docs/contributing/docs.rst
from util.nbmock import mock_datasets, is_nb_fast
if is_nb_fast():
    mock_datasets()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from openmoa.datasets import Electricity, ElectricityTiny
from openmoa.stream.stream_wrapper import (
    OpenFeatureStream,
    TrapezoidalStream,
    CapriciousStream,
    EvolvableStream,
    ShuffledStream,
)

# Use a smaller dataset in fast mode so the notebook runs quickly in CI
NB_FAST = os.environ.get("NB_FAST", "False").lower() == "true"
DatasetCls = ElectricityTiny if NB_FAST else Electricity
N_DEMO = 1_000 if NB_FAST else 10_000   # instances to visualise

print(f"Dataset : {DatasetCls.__name__}")
print(f"NB_FAST : {NB_FAST}")

## Helper: feature-activation heatmap

For every wrapper that changes *which* features are visible, we can track
the active feature set at each time step and render it as a binary heatmap
(blue = active, white = absent/NaN).

In [ ]:
def activation_heatmap(stream, n_instances: int, d_total: int, title: str, ax=None):
    """Consume `n_instances` from `stream` and plot a feature-activation heatmap.

    Works for both OpenFeatureStream (reads `feature_indices`) and
    NaN-filling wrappers (reads `instance.x` and checks for NaN).
    """
    mask = np.zeros((d_total, n_instances), dtype=bool)

    for t in range(n_instances):
        if not stream.has_more_instances():
            break
        inst = stream.next_instance()
        x = np.asarray(inst.x)

        if hasattr(inst, "feature_indices") and inst.feature_indices is not None:
            # OpenFeatureStream: variable-length x with explicit global IDs
            for gid in inst.feature_indices:
                if gid < d_total:
                    mask[gid, t] = True
        else:
            # NaN-filling wrappers: fixed-length x, NaN means absent
            active = ~np.isnan(x)
            mask[: len(active), t] = active

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(10, 3))

    ax.imshow(mask, aspect="auto", cmap="Blues", interpolation="nearest", origin="lower")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Time step")
    ax.set_ylabel("Feature ID")
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    if own_fig:
        plt.tight_layout()
        plt.show()

---
## 1. `OpenFeatureStream` — Variable-Length Output with Global Index Mapping

`OpenFeatureStream` is the flagship wrapper.  At each time step it exposes only the
currently *active* features and attaches a `feature_indices` attribute to every instance —
a list of **global feature IDs** that correspond positionally to the values in `instance.x`.

This solves the *index-shift problem*: a classifier can maintain weight vectors keyed
by global feature ID and remain correct even as the physical array length changes.

### 1.1 Inspecting a single instance

In [ ]:
base = DatasetCls()
d = base.get_schema().get_num_attributes()
n_total = min(N_DEMO, base.get_num_instances() if hasattr(base, "get_num_instances") else N_DEMO)

ofs = OpenFeatureStream(
    base_stream=DatasetCls(),
    evolution_pattern="incremental",   # dimension grows monotonically
    d_min=2,
    total_instances=n_total,
    feature_selection="prefix",
    random_seed=42,
)

inst = ofs.next_instance()
print(f"Global feature IDs : {inst.feature_indices}")
print(f"Feature values     : {inst.x}")
print(f"Label              : {inst.y_index}")
print(f"len(x) == len(feature_indices): {len(inst.x) == len(inst.feature_indices)}")

### 1.2 Six evolution patterns

`OpenFeatureStream` supports six `evolution_pattern` values:

| Pattern | Description |
|---------|-------------|
| `'incremental'` | Dimension grows linearly from `d_min` to `d_max` |
| `'decremental'` | Dimension shrinks linearly from `d_max` to `d_min` |
| `'pyramid'` | Grows to `d_max` then shrinks back to `d_min` |
| `'tds'` | Features have staggered birth times (Trapezoidal Data Stream) |
| `'cds'` | Each feature independently present with probability `1 - missing_ratio` |
| `'eds'` | Feature space rotates through `n_segments` partitions (Evolvable Data Stream) |

In [ ]:
patterns = [
    ("incremental (prefix)",  dict(evolution_pattern="incremental", d_min=2, feature_selection="prefix")),
    ("incremental (random)",  dict(evolution_pattern="incremental", d_min=2, feature_selection="random")),
    ("decremental (prefix)",  dict(evolution_pattern="decremental", d_min=2, feature_selection="prefix")),
    ("pyramid (prefix)",      dict(evolution_pattern="pyramid",     d_min=2, feature_selection="prefix")),
    ("tds (ordered)",         dict(evolution_pattern="tds",         tds_mode="ordered")),
    ("tds (random)",          dict(evolution_pattern="tds",         tds_mode="random")),
    ("cds (missing=0.4)",     dict(evolution_pattern="cds",         missing_ratio=0.4)),
    ("eds (3 segs, ov=0.5)",  dict(evolution_pattern="eds",         n_segments=3, overlap_ratio=0.5)),
]

n_vis = min(500, n_total)
fig, axes = plt.subplots(2, 4, figsize=(16, 5), sharey=True)

for ax, (title, kwargs) in zip(axes.flat, patterns):
    stream = OpenFeatureStream(
        base_stream=DatasetCls(),
        total_instances=n_vis,
        random_seed=42,
        **kwargs,
    )
    activation_heatmap(stream, n_vis, d, title, ax=ax)

fig.suptitle("OpenFeatureStream — feature activation across six evolution patterns",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1.3 `restart()` reproducibility

Calling `restart()` resets both the base stream and the internal RNG so that
the second pass produces an identical sequence of instances.

In [ ]:
n_check = 50
stream = OpenFeatureStream(
    base_stream=DatasetCls(),
    evolution_pattern="cds",
    missing_ratio=0.3,
    total_instances=n_check,
    random_seed=0,
)

first_pass  = [tuple(stream.next_instance().feature_indices) for _ in range(n_check)]
stream.restart()
second_pass = [tuple(stream.next_instance().feature_indices) for _ in range(n_check)]

assert first_pass == second_pass, "restart() did not reproduce the same sequence!"
print("restart() ✓  — both passes produced identical feature_indices sequences.")

---
## 2. `TrapezoidalStream` — Features Phase In Gradually (TDS)

Implements the **Trapezoidal Data Stream** paradigm.  The full feature vector
is always returned (fixed width = `d_max`), but features that have not yet
"been born" are filled with `NaN`.  Features enter the stream in 10 evenly-spaced
stages, either in original order (`evolution_mode='ordered'`) or a random
permutation (`evolution_mode='random'`).

In [ ]:
n_vis = min(500, n_total)
fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)

for ax, mode in zip(axes, ["ordered", "random"]):
    stream = TrapezoidalStream(
        base_stream=DatasetCls(),
        evolution_mode=mode,
        total_instances=n_vis,
        random_seed=42,
    )
    # Verify fixed output width
    inst = stream.next_instance()
    assert len(inst.x) == d, f"Expected width {d}, got {len(inst.x)}"
    stream.restart()

    activation_heatmap(stream, n_vis, d, f"TrapezoidalStream (mode='{mode}')", ax=ax)

plt.suptitle("TrapezoidalStream — NaN-filled fixed-width output", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Show the NaN pattern on the first few instances
stream = TrapezoidalStream(base_stream=DatasetCls(), evolution_mode="ordered",
                           total_instances=n_total, random_seed=42)
print("First instance x :", stream.next_instance().x)
print("(NaN = feature not yet active in TDS)'")

---
## 3. `CapriciousStream` — Random Missing Features (CDS)

At each time step each feature is **independently masked** with probability
`missing_ratio`.  Missing values appear as `NaN`.  The output is fixed-width.

In [ ]:
n_vis = min(500, n_total)
fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)

for ax, ratio in zip(axes, [0.2, 0.5, 0.8]):
    stream = CapriciousStream(
        base_stream=DatasetCls(),
        missing_ratio=ratio,
        total_instances=n_vis,
        random_seed=42,
    )
    activation_heatmap(stream, n_vis, d, f"CapriciousStream (missing_ratio={ratio})", ax=ax)

plt.suptitle("CapriciousStream — independent Bernoulli masking per feature per step",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Verify: missing_ratio=0 produces no NaN
clean = CapriciousStream(DatasetCls(), missing_ratio=0.0, total_instances=200)
inst = clean.next_instance()
assert not np.any(np.isnan(inst.x)), "Expected no NaN when missing_ratio=0"
print("missing_ratio=0 ✓  — no NaN values produced.")

---
## 4. `EvolvableStream` — Segmented Feature Rotation (EDS)

Divides the stream into `n_segments` **stable periods** separated by
**transition periods** of relative length `overlap_ratio`.  During a stable
period only one feature partition is active; during a transition both the
outgoing and incoming partitions are active simultaneously.  Inactive features
are `NaN`.

In [ ]:
n_vis = min(500, n_total)
configs = [
    dict(n_segments=2, overlap_ratio=0.0),
    dict(n_segments=2, overlap_ratio=1.0),
    dict(n_segments=3, overlap_ratio=0.5),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)

for ax, cfg in zip(axes, configs):
    stream = EvolvableStream(
        base_stream=DatasetCls(),
        total_instances=n_vis,
        random_seed=42,
        **cfg,
    )
    title = f"EvolvableStream\n(segs={cfg['n_segments']}, overlap={cfg['overlap_ratio']})"
    activation_heatmap(stream, n_vis, d, title, ax=ax)

plt.suptitle("EvolvableStream — segmented feature rotation (EDS)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5. `ShuffledStream` — Eliminating Label-Sorting Bias

Many benchmark datasets are sorted by class label.  Any learner with a memory
component will appear to perform well simply because similar labels cluster in
time.  `ShuffledStream` buffers the entire dataset in memory and shuffles
instances before replay, giving a fair evaluation.

Feature values and dimensionality are **unchanged** — this wrapper only
reorders instances.

In [ ]:
n_check = min(200, n_total)

# Collect labels from the raw stream
raw = DatasetCls()
labels_raw = [raw.next_instance().y_index for _ in range(n_check)]

# Collect labels from the shuffled stream (same seed → same shuffle)
shuf_a = ShuffledStream(DatasetCls(), random_seed=7)
labels_shuf_a = [shuf_a.next_instance().y_index for _ in range(n_check)]

# Same seed → identical order on restart
shuf_b = ShuffledStream(DatasetCls(), random_seed=7)
labels_shuf_b = [shuf_b.next_instance().y_index for _ in range(n_check)]

assert labels_shuf_a == labels_shuf_b, "Same seed must produce the same shuffle"

# Different seed → different order
shuf_c = ShuffledStream(DatasetCls(), random_seed=99)
labels_shuf_c = [shuf_c.next_instance().y_index for _ in range(n_check)]
assert labels_shuf_a != labels_shuf_c, "Different seeds must produce different shuffles"

fig, axes = plt.subplots(1, 2, figsize=(12, 2))
axes[0].plot(labels_raw,    lw=0.8, color="steelblue")
axes[0].set_title("Raw stream — label sequence");  axes[0].set_ylabel("Class")
axes[1].plot(labels_shuf_a, lw=0.8, color="tomato")
axes[1].set_title("ShuffledStream — label sequence")
for ax in axes:
    ax.set_xlabel("Instance index")
    ax.set_yticks([0, 1])
plt.tight_layout()
plt.show()

print("ShuffledStream ✓  — same seed reproduces order; different seed differs.")

---
## 6. Combining Wrappers

Wrappers can be **composed**: first shuffle to remove label-ordering bias,
then apply a feature-evolution wrapper to simulate dynamic feature spaces.
This is the setup used in the OpenMOA benchmark.

In [ ]:
n_vis = min(500, n_total)

composed = OpenFeatureStream(
    base_stream=ShuffledStream(DatasetCls(), random_seed=42),
    evolution_pattern="eds",
    n_segments=3,
    overlap_ratio=0.3,
    total_instances=n_vis,
    random_seed=42,
)

fig, ax = plt.subplots(figsize=(10, 3))
activation_heatmap(composed, n_vis, d,
                   "ShuffledStream → OpenFeatureStream(eds, 3 segments)", ax=ax)
plt.tight_layout()
plt.show()

---
## Summary

| Wrapper | Key parameter(s) | Output width | NaN fill |
|---------|-----------------|--------------|----------|
| `OpenFeatureStream` | `evolution_pattern`, `d_min`, `n_segments`, `missing_ratio` | **Variable** — reads `feature_indices` | No |
| `TrapezoidalStream` | `evolution_mode` | Fixed = `d_max` | Yes |
| `CapriciousStream` | `missing_ratio` | Fixed = `d_max` | Yes |
| `EvolvableStream` | `n_segments`, `overlap_ratio` | Fixed = `d_max` | Yes |
| `ShuffledStream` | `random_seed` | Unchanged | No |

All wrappers support `restart()` with full RNG reproducibility.
All wrappers compose freely — pass any wrapper as the `base_stream` of another.

To use these streams with a UOL classifier, see **notebook 12: UOL Classifiers**.